In [1]:
import pandas as pd
import csv
import pickle
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from transformers import AutoTokenizer

In [2]:
tokenizer = AutoTokenizer.from_pretrained("dbmdz/bert-base-german-uncased")

In [3]:
path_train = "Input/Data/train/"

test_task1 = pd.read_json(path_train + "test_task1.json")
train_task1 = pd.read_json(path_train + "train_task1.json")

In [4]:
# lists of positive words / emoticons / emojis

path = "Input/Data/"

with open(path + "positive_words.csv", newline='', encoding='utf-8') as infile:
    positive_words = [w for [w] in csv.reader(infile)]

with open(path + "positive_emoticons.csv", newline='', encoding='utf-8') as infile:
    positive_emoticons = [e for [e] in csv.reader(infile)]

with open(path + "positive_emojis.csv", newline='', encoding='utf-8') as infile:
    positive_emojis = [e for [e] in csv.reader(infile)]

# modified list of positive tokens
# see code below
with open(path + "positive_tokens.pkl", "rb") as infile: 
   positive_tokens = pickle.load(infile)

In [5]:
# includes some not flausch but just generally common tokens
positive_tokens = list(set([t for w in positive_words for t in tokenizer.tokenize(w)]))

In [6]:
# clean up positive tokens list

# filter out not_flausch comments from training data
not_flausch_comments = train_task1[train_task1["flausch"] == "no"]

# make dictionary of positive_tokens
token_dict = {}
for t in positive_tokens:
    token_dict[t] = 0

# count how oftem words in dictionary occur in not flausch comments
for i in range(len(not_flausch_comments)):  

    comment = not_flausch_comments.iloc[i]["comment"]
    tokens = tokenizer.tokenize(comment)

    for flausch_token in positive_tokens:
        if flausch_token in tokens:
            token_dict[flausch_token] += 1

# get tokens which occur in more than 2% of not flausch tokens
common_tokens = [key for key, val in token_dict.items() if val > len(not_flausch_comments) / 50]

# delete common tokens from positive tokens list
positive_tokens = [t for t in positive_tokens if t not in common_tokens]

Token indices sequence length is longer than the specified maximum sequence length for this model (2300 > 512). Running this sequence through the model will result in indexing errors


In [7]:
# save modified list of positive tokens
with open(path + "positive_tokens.pkl", "wb") as outfile: 
   pickle.dump(positive_tokens, outfile)

Trying out different features individually

In [8]:
def get01(flausch):
    if flausch == "yes":
        return 1
    else:
        return 0

In [15]:
y_test = []
for i in range(len(test_task1)):

    flausch = test_task1.iloc[i]["flausch"]
    y_test.append(get01(flausch)) 


y_train = []
for i in range(len(train_task1)):

    flausch = train_task1.iloc[i]["flausch"]
    y_train.append(get01(flausch)) 

In [16]:
# simple token count

y_pred = []

for i in range(len(test_task1)):
    
    comment = test_task1.iloc[i]["comment"]
    tokens = tokenizer.tokenize(comment)

    flausch_score = 0

    for t in tokens:
        if t in positive_tokens:
            flausch_score += 1

    # if neutral (score 0), guess not flausch
    guess = "no"
    if flausch_score > 0:
        guess = "yes"
    
    y_pred.append(get01(guess))

print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.87      0.79      0.83      2662
           1       0.57      0.71      0.64      1044

    accuracy                           0.77      3706
   macro avg       0.72      0.75      0.73      3706
weighted avg       0.79      0.77      0.78      3706



In [17]:
# ratio flausch to all tokens

y_pred = []

for i in range(len(test_task1)):
    
    comment = test_task1.iloc[i]["comment"]
    tokens = tokenizer.tokenize(comment)

    flausch_score = 0

    for t in positive_tokens:
        if t in comment:
            flausch_score += 1

    flausch_ratio = flausch_score / len(tokens)

    guess = "no"
    if flausch_ratio > 0.1:
        guess = "yes"
    
    y_pred.append(get01(guess))

print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.79      0.82      0.80      2662
           1       0.49      0.43      0.46      1044

    accuracy                           0.71      3706
   macro avg       0.64      0.63      0.63      3706
weighted avg       0.70      0.71      0.71      3706



In [18]:
# simple word count

y_pred = []

for i in range(len(test_task1)):

    comment = test_task1.iloc[i]["comment"]
    flausch_score = 0

    for w in positive_words:
        if w in comment:
            flausch_score += 1

    # if neutral (score 0), guess not flausch
    guess = "no"
    if flausch_score > 0:
        guess = "yes"
    
    y_pred.append(get01(guess))

print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.82      0.94      0.87      2662
           1       0.75      0.48      0.58      1044

    accuracy                           0.81      3706
   macro avg       0.78      0.71      0.73      3706
weighted avg       0.80      0.81      0.79      3706



In [20]:
# simple emoticon count

y_pred = []

for i in range(len(test_task1)):

    comment = test_task1.iloc[i]["comment"]

    flausch_score = 0

    for e in positive_emoticons:
        if e in comment:
            flausch_score += 1

    # if neutral (score 0), guess not flausch
    guess = "no"
    if flausch_score > 0:
        guess = "yes"
    
    y_pred.append(get01(guess))

print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.74      0.89      0.81      2662
           1       0.42      0.20      0.27      1044

    accuracy                           0.70      3706
   macro avg       0.58      0.55      0.54      3706
weighted avg       0.65      0.70      0.66      3706



In [21]:
# simple emoji count

y_pred = []

for i in range(len(test_task1)):
    
    comment = test_task1.iloc[i]["comment"]

    flausch_score = 0

    for e in positive_emojis:
        if e in comment:
            flausch_score += 1

    # if neutral (score 0), guess not flausch
    guess = "no"
    if flausch_score > 0:
        guess = "yes"
    
    y_pred.append(get01(guess))

print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.72      0.90      0.80      2662
           1       0.33      0.13      0.19      1044

    accuracy                           0.68      3706
   macro avg       0.53      0.51      0.49      3706
weighted avg       0.61      0.68      0.63      3706



In [22]:
y_pred = []

for i in range(len(test_task1)):

    comment = test_task1.iloc[i]["comment"]

    flausch_score = 0

    for e in positive_emojis:
        if e in comment:
            flausch_score += 1
    
    for e in positive_emoticons:
        if e in comment:
            flausch_score += 1
    
    for w in positive_words:
        if w in comment:
            flausch_score += 1

    # if neutral (score 0), guess not flausch
    guess = "no"
    if flausch_score > 0:
        guess = "yes"
    
    y_pred.append(get01(guess))

print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.84      0.75      0.80      2662
           1       0.51      0.64      0.56      1044

    accuracy                           0.72      3706
   macro avg       0.67      0.70      0.68      3706
weighted avg       0.75      0.72      0.73      3706



In [23]:
y_pred = []

for i in range(len(test_task1)):

    comment = test_task1.iloc[i]["comment"]

    flausch_score = 0

    for e in positive_emojis:
        if e in comment:
            flausch_score += 1
    
    for e in positive_emoticons:
        if e in comment:
            flausch_score += 1
    
    for w in positive_words:
        if w in comment:
            flausch_score += 1
    
    for t in positive_tokens:
        if t in comment:
            flausch_score += 1

    # if neutral (score 0), guess not flausch
    guess = "no"
    if flausch_score > 0:
        guess = "yes"
    
    y_pred.append(get01(guess))

print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.84      0.47      0.60      2662
           1       0.36      0.78      0.50      1044

    accuracy                           0.55      3706
   macro avg       0.60      0.62      0.55      3706
weighted avg       0.71      0.55      0.57      3706



In [24]:
def all_caps(comment):

    upper_count = 0

    current = ""
    last_upper = False

    in_upper = False

    for c in comment:

        # idea: notice when at least two uppercase characters in a row -> upper_count up, 
        # but don't continue upping count for every character, only when new word starts (after whitespace)

        current = c
        
        if current.isupper(): # current character is uppercase

            if last_upper: # previous character was uppercase

                if not in_upper: # second character in a row
                    upper_count += 1
                    in_upper = True
            
        else: # current is not uppercase
            in_upper = False # when uppercase character chain ends
        
        last_upper = current.isupper()
    
    return upper_count


In [25]:
y_pred = []

for i in range(len(test_task1)):

    comment = test_task1.iloc[i]["comment"]

    flausch_score = all_caps(comment)

    # if neutral (score 0), guess not flausch
    guess = "no"
    if flausch_score > 0:
        guess = "yes"
    
    y_pred.append(get01(guess))

print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.72      0.93      0.81      2662
           1       0.33      0.09      0.14      1044

    accuracy                           0.69      3706
   macro avg       0.53      0.51      0.48      3706
weighted avg       0.61      0.69      0.62      3706



In [26]:
def extra_chars(comment):

    extra_count = 0 # count how many additional characters there are
    rep_count = 0 # count how often a character is repeated

    current = ""
    last = ""
    
    for c in comment:
        current = c

        if current == last: # repeated character
            rep_count += 1

            if rep_count > 1: # character repeated more than once
                extra_count += 1
        
        else:
            rep_count = 0 # currently no repetition
            last = current
    
    return extra_count

In [27]:
y_pred = []

for i in range(len(test_task1)):

    comment = test_task1.iloc[i]["comment"]

    flausch_score = extra_chars(comment)

    # if neutral (score 0), guess not flausch
    guess = "no"
    if flausch_score > 0:
        guess = "yes"
    
    y_pred.append(get01(guess))

print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.73      0.90      0.81      2662
           1       0.38      0.15      0.22      1044

    accuracy                           0.69      3706
   macro avg       0.55      0.53      0.51      3706
weighted avg       0.63      0.69      0.64      3706



In [28]:
y_pred = []

for i in range(len(test_task1)):

    comment = test_task1.iloc[i]["comment"]

    flausch_score = 0

    for e in positive_emojis:
        if e in comment:
            flausch_score += 1
    
    for e in positive_emoticons:
        if e in comment:
            flausch_score += 1
    
    for w in positive_words:
        if w in comment:
            flausch_score += 1
    
    for t in positive_tokens:
        if t in comment:
            flausch_score += 1

    flausch_score += all_caps(comment)
    flausch_score += extra_chars(comment)

    # if neutral (score 0), guess not flausch
    guess = "no"
    if flausch_score > 0:
        guess = "yes"
    
    y_pred.append(get01(guess))

print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.86      0.41      0.55      2662
           1       0.35      0.83      0.50      1044

    accuracy                           0.53      3706
   macro avg       0.61      0.62      0.53      3706
weighted avg       0.72      0.53      0.54      3706



In [29]:
y_pred = []

for i in range(len(test_task1)):

    sentiment = test_task1.iloc[i]["sentiment_orig"]

    guess = "no"
    if sentiment > 0.2:
        guess = "yes"
    
    y_pred.append(get01(guess))

print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.86      0.86      0.86      2662
           1       0.64      0.64      0.64      1044

    accuracy                           0.80      3706
   macro avg       0.75      0.75      0.75      3706
weighted avg       0.80      0.80      0.80      3706



In [30]:
y_pred = []

for i in range(len(test_task1)):

    sentiment = test_task1.iloc[i]["sentiment_spelling_corrected"]

    guess = "no"
    if sentiment > 0.2:
        guess = "yes"
    
    y_pred.append(get01(guess))

print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.86      0.86      0.86      2662
           1       0.65      0.65      0.65      1044

    accuracy                           0.80      3706
   macro avg       0.76      0.75      0.76      3706
weighted avg       0.80      0.80      0.80      3706



In [31]:
y_pred = []

for i in range(len(test_task1)):

    sentiment = test_task1.iloc[i]["sentiment_translated"]

    guess = "no"
    if sentiment > 0.2:
        guess = "yes"
    
    y_pred.append(get01(guess))

print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.86      0.80      0.83      2662
           1       0.57      0.66      0.61      1044

    accuracy                           0.76      3706
   macro avg       0.71      0.73      0.72      3706
weighted avg       0.78      0.76      0.77      3706



In [32]:
y_pred = []

for i in range(len(test_task1)):

    sentiment_orig = test_task1.iloc[i]["sentiment_orig"]
    sentiment_spelling_corrected = test_task1.iloc[i]["sentiment_spelling_corrected"]
    sentiment_translated = test_task1.iloc[i]["sentiment_translated"]

    sentiment = (sentiment_orig + sentiment_spelling_corrected + sentiment_translated) / 3

    guess = "no"
    if sentiment > 0.2:
        guess = "yes"
    
    y_pred.append(get01(guess))

print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.89      0.83      0.86      2662
           1       0.62      0.74      0.68      1044

    accuracy                           0.80      3706
   macro avg       0.76      0.78      0.77      3706
weighted avg       0.82      0.80      0.81      3706



Trying out logistic regression with different feature combinations

In [33]:
X_test = []
X_train = []

for part in ["train", "test"]:

    if part == "train":
        data = train_task1
        x = X_train
    else:
        data = test_task1
        x = X_test

    for i in range(len(data)):
    
        comment = data.iloc[i]["comment"]
        tokens = tokenizer.tokenize(comment)

        feature_vec = []

        # add sentiments
        feature_vec.append(data.iloc[i]["sentiment_orig"])
        feature_vec.append(data.iloc[i]["sentiment_spelling_corrected"])
        feature_vec.append(data.iloc[i]["sentiment_translated"])
        # sentiment average
        #feature_vec.append((data.iloc[i]["sentiment_orig"] +
        #                    data.iloc[i]["sentiment_spelling_corrected"] +
        #                    data.iloc[i]["sentiment_translated"]) / 3)

        # add word count
        flausch_word_count = 0
        for w in positive_words:
            if w in comment:
                flausch_word_count += 1
        feature_vec.append(flausch_word_count)

        # add token count
        flausch_token_count = 0
        for t in tokens:
            if t in positive_tokens:
                flausch_token_count += 1
        feature_vec.append(flausch_token_count)
        # positive to all token ratio
        feature_vec.append(flausch_token_count/len(tokens))

        # add emoticon count
        emoticon_count = 0
        for e in positive_emoticons:
            if e in comment:
                emoticon_count += 1
        feature_vec.append(emoticon_count)

        # add emoji count
        emoji_count = 0
        for e in positive_emojis:
            if e in comment:
                emoji_count += 1
        feature_vec.append(emoji_count)

        # number of extra characters
        feature_vec.append(extra_chars(comment))
        # number of words in all uppercase
        feature_vec.append(all_caps(comment))

        x.append(feature_vec)

log_reg = LogisticRegression().fit(X_train, y_train)
y_pred = log_reg.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.87      0.93      0.90      2662
           1       0.77      0.63      0.70      1044

    accuracy                           0.84      3706
   macro avg       0.82      0.78      0.80      3706
weighted avg       0.84      0.84      0.84      3706



In [34]:
X_test = []
X_train = []

for part in ["train", "test"]:

    if part == "train":
        data = train_task1
        x = X_train
    else:
        data = test_task1
        x = X_test

    for i in range(len(data)):
    
        comment = data.iloc[i]["comment"]
        tokens = tokenizer.tokenize(comment)

        feature_vec = []

        # add sentiments
        #feature_vec.append(data.iloc[i]["sentiment_orig"])
        #feature_vec.append(data.iloc[i]["sentiment_spelling_corrected"])
        #feature_vec.append(data.iloc[i]["sentiment_translated"])
        # sentiment average
        feature_vec.append((data.iloc[i]["sentiment_orig"] +
                            data.iloc[i]["sentiment_spelling_corrected"] +
                            data.iloc[i]["sentiment_translated"]) / 3)

        # add word count
        flausch_word_count = 0
        for w in positive_words:
            if w in comment:
                flausch_word_count += 1
        feature_vec.append(flausch_word_count)

        # add token count
        flausch_token_count = 0
        for t in tokens:
            if t in positive_tokens:
                flausch_token_count += 1
        feature_vec.append(flausch_token_count)
        # positive to all token ratio
        feature_vec.append(flausch_token_count/len(tokens))

        # add emoticon count
        emoticon_count = 0
        for e in positive_emoticons:
            if e in comment:
                emoticon_count += 1
        feature_vec.append(emoticon_count)

        # add emoji count
        emoji_count = 0
        for e in positive_emojis:
            if e in comment:
                emoji_count += 1
        feature_vec.append(emoji_count)

        # number of extra characters
        feature_vec.append(extra_chars(comment))
        # number of words in all uppercase
        feature_vec.append(all_caps(comment))

        x.append(feature_vec)

log_reg = LogisticRegression().fit(X_train, y_train)
y_pred = log_reg.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.86      0.93      0.89      2662
           1       0.78      0.62      0.69      1044

    accuracy                           0.84      3706
   macro avg       0.82      0.77      0.79      3706
weighted avg       0.84      0.84      0.84      3706



In [35]:
X_test = []
X_train = []

for part in ["train", "test"]:

    if part == "train":
        data = train_task1
        x = X_train
    else:
        data = test_task1
        x = X_test

    for i in range(len(data)):
    
        comment = data.iloc[i]["comment"]
        tokens = tokenizer.tokenize(comment)

        feature_vec = []

        # add sentiments
        feature_vec.append(data.iloc[i]["sentiment_orig"])
        feature_vec.append(data.iloc[i]["sentiment_spelling_corrected"])
        feature_vec.append(data.iloc[i]["sentiment_translated"])
        # sentiment average
        #feature_vec.append((data.iloc[i]["sentiment_orig"] +
        #                    data.iloc[i]["sentiment_spelling_corrected"] +
        #                    data.iloc[i]["sentiment_translated"]) / 3)

        # add word count
        flausch_word_count = 0
        for w in positive_words:
            if w in comment:
                flausch_word_count += 1
        feature_vec.append(flausch_word_count)

        # add token count
        flausch_token_count = 0
        for t in tokens:
            if t in positive_tokens:
                flausch_token_count += 1
        feature_vec.append(flausch_token_count)
        # positive to all token ratio
        feature_vec.append(flausch_token_count/len(tokens))

        # add emoticon count
        emoticon_count = 0
        for e in positive_emoticons:
            if e in comment:
                emoticon_count += 1
        feature_vec.append(emoticon_count)

        # add emoji count
        emoji_count = 0
        for e in positive_emojis:
            if e in comment:
                emoji_count += 1
        feature_vec.append(emoji_count)

        # number of extra characters
        #feature_vec.append(extra_chars(comment))
        # number of words in all uppercase
        #feature_vec.append(all_caps(comment))

        x.append(feature_vec)

log_reg = LogisticRegression().fit(X_train, y_train)
y_pred = log_reg.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.86      0.93      0.90      2662
           1       0.77      0.63      0.69      1044

    accuracy                           0.84      3706
   macro avg       0.82      0.78      0.79      3706
weighted avg       0.84      0.84      0.84      3706



In [ ]:
X_test = []
X_train = []

for part in ["train", "test"]:

    if part == "train":
        data = train_task1
        x = X_train
    else:
        data = test_task1
        x = X_test

    for i in range(len(data)):
    
        comment = data.iloc[i]["comment"]
        tokens = tokenizer.tokenize(comment)

        feature_vec = []

        # add sentiments
        feature_vec.append(data.iloc[i]["sentiment_orig"])
        feature_vec.append(data.iloc[i]["sentiment_spelling_corrected"])
        feature_vec.append(data.iloc[i]["sentiment_translated"])
        # sentiment average
        #feature_vec.append((data.iloc[i]["sentiment_orig"] +
        #                    data.iloc[i]["sentiment_spelling_corrected"] +
        #                    data.iloc[i]["sentiment_translated"]) / 3)

        # add word count
        flausch_word_count = 0
        for w in positive_words:
            if w in comment:
                flausch_word_count += 1
        feature_vec.append(flausch_word_count)

        # add token count
        flausch_token_count = 0
        for t in tokens:
            if t in positive_tokens:
                flausch_token_count += 1
        feature_vec.append(flausch_token_count)
        # positive to all token ratio
        feature_vec.append(flausch_token_count/len(tokens))

        # add emoticon count
        #emoticon_count = 0
        #for e in positive_emoticons:
        #    if e in comment:
        #        emoticon_count += 1
        #feature_vec.append(emoticon_count)

        # add emoji count
        #emoji_count = 0
        #for e in positive_emojis:
        #    if e in comment:
        #        emoji_count += 1
        #feature_vec.append(emoji_count)

        # number of extra characters
        #feature_vec.append(extra_chars(comment))
        # number of words in all uppercase -> leaving out gets slightly worse
        feature_vec.append(all_caps(comment))

        x.append(feature_vec)

log_reg = LogisticRegression().fit(X_train, y_train)
y_pred = log_reg.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.86      0.93      0.90      2662
           1       0.78      0.63      0.70      1044

    accuracy                           0.85      3706
   macro avg       0.82      0.78      0.80      3706
weighted avg       0.84      0.85      0.84      3706



Adding features to features.csv

In [37]:
features = pd.read_csv(path_train + "features.csv")
test_features = features[features["set"] == "test"].copy()

In [38]:
sentiment_orig = []
sentiment_spelling_corrected = []
sentiment_translated = []
flausch_words = []
flausch_tokens = []
flausch_ratio = []
flausch_emoticons = []
flausch_emojis = []
extra_characters = []
all_capitalized = []

In [39]:
for i in range(len(test_features)):

    id = test_features.iloc[i]["id"]
    
    comment = test_task1[test_task1["id"] == id]["comment"].item()
    tokens = tokenizer.tokenize(comment)

    # add sentiments
    sentiment_orig.append(test_task1[test_task1["id"] == id]["sentiment_orig"].item())
    sentiment_spelling_corrected.append(test_task1[test_task1["id"] == id]["sentiment_spelling_corrected"].item())
    sentiment_translated.append(test_task1[test_task1["id"] == id]["sentiment_translated"].item())

    # add word count
    flausch_word_count = 0
    for w in positive_words:
        if w in comment:
            flausch_word_count += 1
    flausch_words.append(flausch_word_count)

    # add token count
    flausch_token_count = 0
    for t in tokens:
        if t in positive_tokens:
            flausch_token_count += 1
    flausch_tokens.append(flausch_token_count)
    # positive to all token ratio
    flausch_ratio.append(flausch_token_count/len(tokens))

    # add emoticon count
    emoticon_count = 0
    for e in positive_emoticons:
        if e in comment:
            emoticon_count += 1
    flausch_emoticons.append(emoticon_count)

    # add emoji count
    emoji_count = 0
    for e in positive_emojis:
        if e in comment:
            emoji_count += 1
    flausch_emojis.append(emoji_count)

    # number of extra characters
    extra_characters.append(extra_chars(comment))
    # number of words in all uppercase
    all_capitalized.append(all_caps(comment))

In [40]:
test_features["sentiment_orig"] = sentiment_orig
test_features["sentiment_spelling_corrected"] = sentiment_spelling_corrected
test_features["sentiment_translated"] = sentiment_translated
test_features["flausch_words"] = flausch_words
test_features["flausch_tokens"] = flausch_tokens
test_features["flausch_ratio"] = flausch_ratio
test_features["flausch_emoticons"] = flausch_emoticons
test_features["flausch_emojis"] = flausch_emojis
test_features["extra_chars"] = extra_characters
test_features["all_caps"] = all_capitalized

In [41]:
test_features.to_csv(path_train + "test_features.csv", index=False)

Logistic regression for all features (including BERT)

In [42]:
X = []
y = []

for i in range(len(test_features)):

    id = test_features.iloc[i]["id"]
    flausch = test_task1[test_task1["id"] == id]["flausch"].item()
    y.append(get01(flausch))

    feature_vec = []
    for c in list(test_features.columns)[3:]:
        feature_vec.append(test_features.iloc[i][c])
    X.append(feature_vec)


In [43]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test =  train_test_split(X, y, test_size=0.2, random_state=42)

In [44]:
log_reg = LogisticRegression().fit(X_train, y_train)
y_pred = log_reg.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.97      0.98      0.97       536
           1       0.94      0.93      0.93       206

    accuracy                           0.96       742
   macro avg       0.95      0.95      0.95       742
weighted avg       0.96      0.96      0.96       742



In [45]:
X = []
y = []

for i in range(len(test_features)):

    id = test_features.iloc[i]["id"]
    flausch = test_task1[test_task1["id"] == id]["flausch"].item()
    y.append(get01(flausch))

    feature_vec = []
    for c in list(test_features.columns)[3:8]: # just BERT
        feature_vec.append(test_features.iloc[i][c])
    X.append(feature_vec)

X_train, X_test, y_train, y_test =  train_test_split(X, y, test_size=0.2, random_state=42)

log_reg = LogisticRegression().fit(X_train, y_train)
y_pred = log_reg.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.97      0.98      0.97       536
           1       0.94      0.91      0.93       206

    accuracy                           0.96       742
   macro avg       0.95      0.94      0.95       742
weighted avg       0.96      0.96      0.96       742



In [46]:
X = []
y = []

for i in range(len(test_features)):

    id = test_features.iloc[i]["id"]
    flausch = test_task1[test_task1["id"] == id]["flausch"].item()
    y.append(get01(flausch))

    feature_vec = []
    for c in list(test_features.columns)[3:11]: # BERT & sentiments
        feature_vec.append(test_features.iloc[i][c])
    X.append(feature_vec)

X_train, X_test, y_train, y_test =  train_test_split(X, y, test_size=0.2, random_state=42)

log_reg = LogisticRegression().fit(X_train, y_train)
y_pred = log_reg.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.96      0.98      0.97       536
           1       0.94      0.90      0.92       206

    accuracy                           0.96       742
   macro avg       0.95      0.94      0.95       742
weighted avg       0.96      0.96      0.96       742



In [47]:
X = []
y = []

for i in range(len(test_features)):

    id = test_features.iloc[i]["id"]
    flausch = test_task1[test_task1["id"] == id]["flausch"].item()
    y.append(get01(flausch))

    feature_vec = []
    for c in list(test_features.columns)[3:14]: # BERT, sentiments, flausch word count, token count, ratio
        feature_vec.append(test_features.iloc[i][c])
    X.append(feature_vec)

X_train, X_test, y_train, y_test =  train_test_split(X, y, test_size=0.2, random_state=42)

log_reg = LogisticRegression().fit(X_train, y_train)
y_pred = log_reg.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.97      0.98      0.97       536
           1       0.94      0.91      0.93       206

    accuracy                           0.96       742
   macro avg       0.95      0.95      0.95       742
weighted avg       0.96      0.96      0.96       742



In [48]:
X = []
y = []

for i in range(len(test_features)):

    id = test_features.iloc[i]["id"]
    flausch = test_task1[test_task1["id"] == id]["flausch"].item()
    y.append(get01(flausch))

    feature_vec = []
    for c in [
            "gbert-large_yes_comment",
            "bert-base-german-cased_yes_comment",
            "bert-base-german-cased_yes_spelling_corrected",
            "gbert-large_yes_spelling_corrected",
            "roberta-large_yes_translated",
            "flausch_words",
            "flausch_tokens",
            "flausch_ratio"
            ]: 
        feature_vec.append(test_features.iloc[i][c])
    X.append(feature_vec)

X_train, X_test, y_train, y_test =  train_test_split(X, y, test_size=0.2, random_state=42)

log_reg = LogisticRegression().fit(X_train, y_train)
y_pred = log_reg.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.97      0.98      0.97       536
           1       0.94      0.91      0.93       206

    accuracy                           0.96       742
   macro avg       0.95      0.95      0.95       742
weighted avg       0.96      0.96      0.96       742



In [49]:
X = []
y = []

for i in range(len(test_features)):

    id = test_features.iloc[i]["id"]
    flausch = test_task1[test_task1["id"] == id]["flausch"].item()
    y.append(get01(flausch))

    feature_vec = []
    for c in [
            "gbert-large_yes_comment",
            "bert-base-german-cased_yes_comment",
            "bert-base-german-cased_yes_spelling_corrected",
            "gbert-large_yes_spelling_corrected",
            "roberta-large_yes_translated",
            "flausch_words",
            "flausch_tokens",
            "flausch_ratio",
            "all_caps"
            ]: 
        feature_vec.append(test_features.iloc[i][c])
    X.append(feature_vec)

X_train, X_test, y_train, y_test =  train_test_split(X, y, test_size=0.2, random_state=42)

log_reg = LogisticRegression().fit(X_train, y_train)
y_pred = log_reg.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.97      0.98      0.97       536
           1       0.94      0.91      0.93       206

    accuracy                           0.96       742
   macro avg       0.95      0.95      0.95       742
weighted avg       0.96      0.96      0.96       742



In [50]:
X = []
y = []

for i in range(len(test_features)):

    id = test_features.iloc[i]["id"]
    flausch = test_task1[test_task1["id"] == id]["flausch"].item()
    y.append(get01(flausch))

    feature_vec = []
    for c in [
            "gbert-large_yes_comment",
            "bert-base-german-cased_yes_comment",
            "bert-base-german-cased_yes_spelling_corrected",
            "gbert-large_yes_spelling_corrected",
            "roberta-large_yes_translated",
            "flausch_words",
            "flausch_tokens",
            "flausch_ratio"
            ]: 
        feature_vec.append(test_features.iloc[i][c])

    feature_vec.append((test_features.iloc[i]["sentiment_orig"] +
                        test_features.iloc[i]["sentiment_spelling_corrected"] +
                        test_features.iloc[i]["sentiment_translated"]) / 3)

    X.append(feature_vec)

X_train, X_test, y_train, y_test =  train_test_split(X, y, test_size=0.2, random_state=42)

log_reg = LogisticRegression().fit(X_train, y_train)
y_pred = log_reg.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.97      0.98      0.97       536
           1       0.94      0.91      0.93       206

    accuracy                           0.96       742
   macro avg       0.95      0.95      0.95       742
weighted avg       0.96      0.96      0.96       742



In [ ]:
# best configuration

X = []
y = []

for i in range(len(test_features)):

    id = test_features.iloc[i]["id"]
    flausch = test_task1[test_task1["id"] == id]["flausch"].item()
    y.append(get01(flausch))

    feature_vec = []
    for c in [
            "gbert-large_yes_comment",
            #"bert-base-german-cased_yes_comment",
            #"bert-base-german-cased_yes_spelling_corrected",
            #"gbert-large_yes_spelling_corrected",
            #"roberta-large_yes_translated",
            "sentiment_orig", 
            "sentiment_spelling_corrected",
            "sentiment_translated",
            "flausch_words",
            "flausch_tokens",
            "flausch_ratio"
            ]: 
        feature_vec.append(test_features.iloc[i][c])
    X.append(feature_vec)

X_train, X_test, y_train, y_test =  train_test_split(X, y, test_size=0.2, random_state=42)

log_reg = LogisticRegression().fit(X_train, y_train)
y_pred = log_reg.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.98      0.97      0.97       536
           1       0.92      0.95      0.94       206

    accuracy                           0.96       742
   macro avg       0.95      0.96      0.95       742
weighted avg       0.96      0.96      0.96       742



In [53]:
X = []
y = []

for i in range(len(test_features)):

    id = test_features.iloc[i]["id"]
    flausch = test_task1[test_task1["id"] == id]["flausch"].item()
    y.append(get01(flausch))

    feature_vec = []
    for c in [
            #"gbert-large_yes_comment",
            "bert-base-german-cased_yes_comment",
            #"bert-base-german-cased_yes_spelling_corrected",
            #"gbert-large_yes_spelling_corrected",
            #"roberta-large_yes_translated",
            "sentiment_orig", 
            "sentiment_spelling_corrected",
            "sentiment_translated",
            "flausch_words",
            "flausch_tokens",
            "flausch_ratio"
            ]: 
        feature_vec.append(test_features.iloc[i][c])
    X.append(feature_vec)

X_train, X_test, y_train, y_test =  train_test_split(X, y, test_size=0.2, random_state=42)

log_reg = LogisticRegression().fit(X_train, y_train)
y_pred = log_reg.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.96      0.97      0.96       536
           1       0.91      0.90      0.90       206

    accuracy                           0.95       742
   macro avg       0.94      0.93      0.93       742
weighted avg       0.95      0.95      0.95       742



In [54]:
X = []
y = []

for i in range(len(test_features)):

    id = test_features.iloc[i]["id"]
    flausch = test_task1[test_task1["id"] == id]["flausch"].item()
    y.append(get01(flausch))

    feature_vec = []
    for c in [
            #"gbert-large_yes_comment",
            #"bert-base-german-cased_yes_comment",
            "bert-base-german-cased_yes_spelling_corrected",
            #"gbert-large_yes_spelling_corrected",
            #"roberta-large_yes_translated",
            "sentiment_orig", 
            "sentiment_spelling_corrected",
            "sentiment_translated",
            "flausch_words",
            "flausch_tokens",
            "flausch_ratio"
            ]: 
        feature_vec.append(test_features.iloc[i][c])
    X.append(feature_vec)

X_train, X_test, y_train, y_test =  train_test_split(X, y, test_size=0.2, random_state=42)

log_reg = LogisticRegression().fit(X_train, y_train)
y_pred = log_reg.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.96      0.95      0.95       536
           1       0.88      0.88      0.88       206

    accuracy                           0.93       742
   macro avg       0.92      0.92      0.92       742
weighted avg       0.93      0.93      0.93       742



In [55]:
X = []
y = []

for i in range(len(test_features)):

    id = test_features.iloc[i]["id"]
    flausch = test_task1[test_task1["id"] == id]["flausch"].item()
    y.append(get01(flausch))

    feature_vec = []
    for c in [
            #"gbert-large_yes_comment",
            #"bert-base-german-cased_yes_comment",
            #"bert-base-german-cased_yes_spelling_corrected",
            "gbert-large_yes_spelling_corrected",
            #"roberta-large_yes_translated",
            "sentiment_orig", 
            "sentiment_spelling_corrected",
            "sentiment_translated",
            "flausch_words",
            "flausch_tokens",
            "flausch_ratio"
            ]: 
        feature_vec.append(test_features.iloc[i][c])
    X.append(feature_vec)

X_train, X_test, y_train, y_test =  train_test_split(X, y, test_size=0.2, random_state=42)

log_reg = LogisticRegression().fit(X_train, y_train)
y_pred = log_reg.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.96      0.96      0.96       536
           1       0.89      0.90      0.89       206

    accuracy                           0.94       742
   macro avg       0.93      0.93      0.93       742
weighted avg       0.94      0.94      0.94       742



In [56]:
X = []
y = []

for i in range(len(test_features)):

    id = test_features.iloc[i]["id"]
    flausch = test_task1[test_task1["id"] == id]["flausch"].item()
    y.append(get01(flausch))

    feature_vec = []
    for c in [
            #"gbert-large_yes_comment",
            #"bert-base-german-cased_yes_comment",
            #"bert-base-german-cased_yes_spelling_corrected",
            #"gbert-large_yes_spelling_corrected",
            "roberta-large_yes_translated",
            "sentiment_orig", 
            "sentiment_spelling_corrected",
            "sentiment_translated",
            "flausch_words",
            "flausch_tokens",
            "flausch_ratio"
            ]: 
        feature_vec.append(test_features.iloc[i][c])
    X.append(feature_vec)

X_train, X_test, y_train, y_test =  train_test_split(X, y, test_size=0.2, random_state=42)

log_reg = LogisticRegression().fit(X_train, y_train)
y_pred = log_reg.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.95      0.96      0.96       536
           1       0.90      0.88      0.89       206

    accuracy                           0.94       742
   macro avg       0.92      0.92      0.92       742
weighted avg       0.94      0.94      0.94       742

